# CCT (Cortical Column Transformer) 本地实验

8GB VRAM 本地验证:
1. 模型构建 + 参数量统计
2. Dummy forward/backward 验证
3. 小批量训练 (batch=1, seq=512, fp16, gradient checkpointing)
4. 循环次数 + precision 分布可视化

In [ ]:
import sys
sys.path.insert(0, '..')

import torch
import torch.nn as nn
from transformers import LlamaForCausalLM, AutoTokenizer

from src.model.column_config import CCTConfig
from src.model.wrapped_model import CCTLlamaModel

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f'Device: {device}')
if device == 'cuda':
    print(f'GPU: {torch.cuda.get_device_name()}')
    print(f'VRAM: {torch.cuda.get_device_properties(0).total_mem / 1e9:.1f} GB')

## 1. 构建模型

In [ ]:
config = CCTConfig(
    model_name='unsloth/Llama-3.2-1B',
    max_iter=10,
    precision_temperature=0.5,
    halt_tau_start=1.0,
    halt_tau_end=0.01,
    fp16=True,
    gradient_checkpointing=True,
    max_seq_len=512,
)

print('Loading base model...')
base_model = LlamaForCausalLM.from_pretrained(
    config.model_name,
    torch_dtype=torch.float16,
    device_map='cpu',
)

print('Building CCT model...')
model = CCTLlamaModel(base_model, config)
model = model.to(device)
model.enable_gradient_checkpointing()

print(model.get_trainable_params_info())

## 2. Dummy Forward + Backward

In [ ]:
# dummy input
batch_size = 1
seq_len = 128  # 小一点先测试

input_ids = torch.randint(0, 32000, (batch_size, seq_len), device=device)
labels = torch.randint(0, 32000, (batch_size, seq_len), device=device)
attention_mask = torch.ones(batch_size, seq_len, dtype=torch.long, device=device)

model.train()
with torch.cuda.amp.autocast(enabled=True):
    outputs = model(
        input_ids=input_ids,
        attention_mask=attention_mask,
        labels=labels,
    )

loss = outputs['loss']
print(f"Loss: {loss.item():.4f}")
print(f"Loss dict: {outputs['loss_dict']}")
print(f"Num iterations: {outputs['num_iterations']}")
print(f"p_halts: {[p.mean().item() for p in outputs['p_halts']]}")

# Backward
loss.backward()
print('\nBackward successful!')

# 检查梯度
for name, p in model.named_parameters():
    if p.grad is not None and p.grad.abs().sum() > 0:
        if 'predictor' in name or 'anchor' in name or 'halt' in name or 'precision' in name:
            print(f'{name}: grad norm = {p.grad.norm().item():.6f}')

model.zero_grad()

## 3. 小批量训练测试

In [ ]:
from src.training.scheduler import compute_halt_tau

# 简单训练循环
optimizer = torch.optim.AdamW(model.get_param_groups(), weight_decay=0.01)
scaler = torch.cuda.amp.GradScaler()

# 用 tokenizer 生成真实数据
tokenizer = AutoTokenizer.from_pretrained(config.model_name)
text = "The quick brown fox jumps over the lazy dog. " * 50
tokens = tokenizer.encode(text, return_tensors='pt')[:, :seq_len].to(device)

num_steps = 20
losses = []
iterations_per_step = []

model.train()
for step in range(num_steps):
    tau_halt = compute_halt_tau(step, num_steps, 1.0, 0.1)
    model.set_halt_tau(tau_halt)
    
    with torch.cuda.amp.autocast(enabled=True):
        outputs = model(
            input_ids=tokens,
            labels=tokens,
        )
    
    loss = outputs['loss']
    scaler.scale(loss).backward()
    scaler.unscale_(optimizer)
    torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad()
    
    losses.append(loss.item())
    iterations_per_step.append(outputs['num_iterations'])
    
    if step % 5 == 0:
        print(f'Step {step}: loss={loss.item():.4f}, iters={outputs["num_iterations"]}, τ_halt={tau_halt:.3f}')

print(f'\nFinal loss: {losses[-1]:.4f}')
print(f'Avg iterations: {sum(iterations_per_step)/len(iterations_per_step):.1f}')

## 4. 可视化

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

fig, axes = plt.subplots(1, 3, figsize=(15, 4))

# Loss 曲线
axes[0].plot(losses)
axes[0].set_xlabel('Step')
axes[0].set_ylabel('Loss')
axes[0].set_title('Training Loss')

# 迭代次数
axes[1].plot(iterations_per_step)
axes[1].set_xlabel('Step')
axes[1].set_ylabel('Iterations')
axes[1].set_title('Column Loop Iterations')

# p_halt 分布 (最后一步)
if outputs['p_halts']:
    p_halt_vals = [p.mean().item() for p in outputs['p_halts']]
    axes[2].bar(range(len(p_halt_vals)), p_halt_vals)
    axes[2].set_xlabel('Iteration')
    axes[2].set_ylabel('p_halt (mean)')
    axes[2].set_title('Halt Probabilities (last step)')
    axes[2].axhline(y=0.5, color='r', linestyle='--', alpha=0.5)

plt.tight_layout()
plt.savefig('../results/local_experiment.png', dpi=150)
plt.show()

print('Figure saved to results/local_experiment.png')

## 5. VRAM 使用统计

In [ ]:
if torch.cuda.is_available():
    print(f'VRAM allocated: {torch.cuda.memory_allocated() / 1e9:.2f} GB')
    print(f'VRAM reserved:  {torch.cuda.memory_reserved() / 1e9:.2f} GB')
    print(f'VRAM max:       {torch.cuda.max_memory_allocated() / 1e9:.2f} GB')

In [ ]:
# 清理显存
del model, optimizer, scaler
torch.cuda.empty_cache()
print('Cleaned up.')